# Track 1 — Legacy Well Log Digitisation

Optional inspection and validation starter. **Not** an OCR, curve extraction or calibration solution. Use the approved dataset provided by the organisers. The notebook locates it when run from the repository root or this starter-kit directory; otherwise set `DATA_ROOT` to its location.

In [ ]:
from pathlib import Path
import csv
from PIL import Image
from IPython.display import display

# Resolve data from either the repository root or the starter-kit directory.
def find_data_root():
    candidates = [
        Path.cwd() / "data" / "track_1",
        Path.cwd() / "../../data/track_1",
        Path.cwd() / "../data/track_1",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate data/track_1. Run from the repository root or the track starter-kit "
        "directory, or set DATA_ROOT to the approved dataset location."
    )

DATA_ROOT = find_data_root()
METADATA_DIR = DATA_ROOT / "metadata"
TEMPLATE = METADATA_DIR / "target_depths_template.csv"
IMAGE_DIR = DATA_ROOT / "log_images"
EXPECTED_COLUMNS = ["page_id", "well", "depth_m", "curve", "value", "confidence"]
print("Dataset found:", DATA_ROOT.exists())
print("Template found:", TEMPLATE.exists())
print("Images folder found:", IMAGE_DIR.exists())

In [ ]:
targets = []
if TEMPLATE.is_file():
    with TEMPLATE.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        actual_columns = reader.fieldnames or []
        missing = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
        print("Template columns:", actual_columns)
        print("Missing required columns:", missing)
        if not missing:
            targets = list(reader)
            print("Target rows:", len(targets))
            print("Sample target rows:", targets[:3])
else:
    print("No template at the placeholder path. Ask technical support for the approved data root.")

In [ ]:
images = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
) if IMAGE_DIR.is_dir() else []
print("Raster image files:", len(images))
if images:
    try:
        with Image.open(images[0]) as image:
            print("First image:", images[0].name, image.size, image.mode)
            display(image.copy())
    except (OSError, ValueError) as exc:
        print("Image could not be previewed:", exc)
else:
    print("No raster images found at this path; supplied PDFs may require separate local processing.")

## Output-contract validator

Once **your own extraction code** has produced a CSV, set `PREDICTIONS` to its location and run this check. It checks structure and target coverage, **not extraction accuracy**.

In [ ]:
PREDICTIONS = Path("outputs/track1_predictions.csv")  # set after implementing your solution
if PREDICTIONS.is_file() and targets:
    with PREDICTIONS.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
        print("Exact output header:", reader.fieldnames == EXPECTED_COLUMNS)
    key = lambda row: (row["page_id"], row["well"], row["depth_m"], row["curve"])
    expected_keys = [key(row) for row in targets]
    output_keys = [key(row) for row in rows]
    print("Row count matches:", len(rows) == len(targets))
    print("Target identifiers match (including duplicates):", sorted(output_keys) == sorted(expected_keys))
else:
    print("Validator skipped: supply your own output CSV and the approved template first.")

## Your implementation

Implement image/PDF handling, pixel-to-depth/value calibration, curve tracing, QC, confidence and an inference entry point. Freeze fitted/calibrated artefacts before submission; ensure your submitted notebook is reproducible. See the Technical Guide and Submission Guide.